In [1]:
# Imports
import os
import sys
from pathlib import Path

# Resolve project root (works when cwd is repo root, examples/, or elsewhere under the repo)
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "katabatic").is_dir():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError(
        "Could not find the Katabatic repo root (no katabatic/ package directory). "
        "Open the notebook from the repository or set the kernel's working directory to the repo root."
    )

# Match cwd to repo root before importing GANBLR (its module mutates sys.path from ".")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_tabular
from katabatic.models.ganblr.models import GANBLR


/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The pipeline registers the logical dataset name (`dataset_name`) automatically on `run()` (`DatasetRegistry.register_if_absent`). No separate registration step is required.

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "car.csv"
output_path = ROOT / "preprocessed_data" / "car.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

preprocess_tabular(str(dataset_path), str(output_path))



Preprocessing: /Users/vikumdabare/Documents/Projects/Katabatic/raw_data/car.csv
Saved preprocessed discrete dataset to: /Users/vikumdabare/Documents/Projects/Katabatic/preprocessed_data/car.csv


In [3]:
# Run pipeline (artifacts under ROOT/artifacts: datasets/, models/, evaluations/, registry/)
input_csv = str(output_path)

from katabatic.artifacts import LocalArtifactStore

artifact_store = LocalArtifactStore(ROOT / "artifacts")

pipeline = TrainTestSplitPipeline(model=GANBLR)
# Registers "car" in registry/datasets.json if missing; require_registered_dataset runs model/dataset compatibility checks.
result = pipeline.run(
    input_csv=input_csv,
    dataset_name="car",
    artifact_store=artifact_store,
    model_name="ganblr",
    require_registered_dataset=True,
    train_epochs=5,
)
print(result)


Loaded data with shape: (1728, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car/split-20260521-161143
Loaded X shape: (1382, 6), y shape: (1382,)
[GANBLR] Importing TensorFlow (first load often takes 1–3+ minutes on macOS; wait for the next line — not frozen).


[GANBLR] TensorFlow ready.
GANBLR: warmup (encoder + generator)…


/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


GANBLR: warmup done.


Generating for node: 2: 100%|██████████| 7/7 [00:00<00:00, 270.50it/s]



 Synthetic data saved to: /Users/vikumdabare/Documents/Projects/Katabatic/artifacts/models/ganblr_car_train-20260521-161143/synthetic
[GANBLR] Saved fitted model state to: /Users/vikumdabare/Documents/Projects/Katabatic/artifacts/models/ganblr_car_train-20260521-161143/state/ganblr_model.pkl


/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [02:12:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: /Users/vikumdabare/Documents/Projects/Katabatic/artifacts/evaluations/ganblr_car_train-20260521-161143/tstr_report.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1 Score: 0.5757

MLP:
Accuracy: 0.7023
F1 Score: 0.5824

RF:
Accuracy: 0.6705
F1 Score: 0.6263

XGBoost:
Accuracy: 0.6532
F1 Score: 0.6176
{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='car', dataset_version='split-20260521-161143'), 'model_ref': ModelRef(model_name='ganblr', dataset_name='car', dataset_version='split-20260521-161143', train_run_id='train-20260521-161143'), 'evaluation_refs': [EvaluationRef(evaluation_type='tstr', eval_run_id='eval-20260521-161203', model_name='ganblr', dataset_name='car', dataset_version='split-20260521-161143', train_run_id='train-20260521-161143', test_dataset_version='split-20260521-161143')]}


In [4]:
# --- TSTR only: reuse the trained run's synthetic data (no re-training) ---
# Requires `result`, `artifact_store`, and `ROOT` from the cells above.
# Writes under artifacts/evaluations/<model>_<dataset>_<train_run_id>/ (e.g. tstr_report.csv, tstr_metrics.json).

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

mr = result["model_ref"]
ds = result["dataset_ref"]

evaluator, eval_ref = TSTREvaluation.from_artifact(artifact_store, mr, ds)
rerun_metrics = evaluator.evaluate()
print("Evaluation directory:", eval_ref.root_relpath)
print("Report:", eval_ref.report_relpath)

/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/vikumdabare/Documents/Projects/Katabatic/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [02:12:05] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: /Users/vikumdabare/Documents/Projects/Katabatic/artifacts/evaluations/ganblr_car_train-20260521-161143/tstr_report.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1 Score: 0.5757

MLP:
Accuracy: 0.6994
F1 Score: 0.5757

RF:
Accuracy: 0.6734
F1 Score: 0.6257

XGBoost:
Accuracy: 0.6532
F1 Score: 0.6176
Evaluation directory: evaluations/ganblr_car_train-20260521-161143
Report: evaluations/ganblr_car_train-20260521-161143/tstr_report.csv
